# Functional Test 1: Entity Graph

## Summary

- **The Foundry IQ service itself builds the Entity Graph.** When you enable `graphExtraction` on a knowledge source, the managed ingestion pipeline inside the service performs chunking, vectorization and triple (subject, predicate, object) extraction. The plugin's Python code (`entity-graph-build`) only sends the definition JSON over REST and watches the status. This notebook calls the service directly from the SDK and **tests the service's features**.
- The input is two text files in the same folder (`Thornwick Castle - Hanapedia.txt` and `Elowen Park - Hanapedia.txt`). The notebook does not depend on other features (Topic Graph, ontology generation). By default it uses the small JSON-LD inside this notebook. You can optionally specify an external JSON-LD, but running Functional Test 3 is not required.
- The procedure is the same as the plugin's "attach the ontology later" flow. **Create the knowledge source without an ontology** (chunking and vectorization only), then **attach the ontology afterward** and run only the triple extraction that reuses the cache.
- Extraction is done by an LLM, so results vary. The verdict is therefore based on the **recall** of the "expected facts" (threshold `MIN_RECALL`) and on the presence of a node shared by the two articles.
- **The default is a new build** (`RUN_BUILD = True`). To only read an existing graph, set `RUN_BUILD = False`. In reuse mode, even in a new kernel, you can go from setup and T1 to the search, drawing and provenance checks from T9 onward. T2 to T8 are not run, and the notebook does not delete existing resources, upload, sync or update the ontology.
- Set `RUN_BUILD = True` only when you verify from a new build. The objects it creates have names that start with `ft-entity`. **T4 deletes the knowledge base and knowledge source with the same names and recreates them.**
- `Run All` runs in the selected mode, and only the tests that ran are recorded as PASS / FAIL. T10 to T12 target the graph already built from this notebook's two articles and ontology.

## Run order from generation to visualization

1. Run chapters 1 to 10 in order with `RUN_BUILD = True`. This uploads the two articles, chunks them, attaches the ontology and extracts relationships. T4 deletes resources with the same names, so check the target and the names first.
2. Chapters 11 to 13 verify the expected facts of the built-in schema, the nodes shared across documents and the evidence. If an external schema does not provide the required types and relationships, these fixed tests are skipped explicitly.
3. Chapter 15 **draws the ontology re-fetched from Foundry IQ together with the actual entities and relationships**, and chapter 16 checks GQL exploration, aggregation and evidence text. The notebook does not re-ingest anything for visualization.

## Shortest path to resume from an existing graph

1. In a new kernel, run the settings, the test schema, the search helpers and the drawing helpers in "1. Setup". Install the packages only if they are not installed yet.
2. Keep `RUN_BUILD = False` and run "2. T1: Connection". The `KS_NAME` to reuse is `ft-entity-ks` by default. To use a different graph, change the name in the settings cell.
3. Run the cells in order starting from "10. T9: Graph index". The graph name and index name come from the latest definition of the existing knowledge source. You can also skip T10 and run T11 and T12.

`Run All` runs the same reuse path. If you only want to draw, go straight from setup and T1 to **chapter 15 and then chapter 16**. You do not need the input articles, an external JSON-LD or build artifacts. If the existing graph cannot be retrieved, the notebook stops with an error and does not switch to a new build automatically.

> Source: [Foundry IQ: Introducing knowledge graphs, engineered for agents](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/foundry-iq-introducing-knowledge-graphs-engineered-for-agents/4572683) / [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md) / [azure-search-documents 12.2.0b1](https://pypi.org/project/azure-search-documents/12.2.0b1/) / [Documents - Graph(REST API 2026-11-01-preview)](https://learn.microsoft.com/en-us/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

## Created resources and related components

**The notebook creates the knowledge source, and Foundry IQ automatically generates the ingestion and search components under it.** Blue marks what the notebook creates or configures, green marks what Foundry IQ generates automatically, and gray marks what you prepare in advance. Solid lines show the main processing and data flow, and dashed lines show configuration, references and model calls.

```mermaid
flowchart TB
    subgraph SOURCE["Source storage (existing account)"]
        BLOB[("Blob container: ft-entity<br/>T4: create or reuse, and place the 2 articles")]
    end

    subgraph SEARCH["Foundry IQ (existing service)"]
        KS["Knowledge source: ft-entity-ks<br/>T5: create with graphExtraction + cache<br/>T8: add the inline ontology"]
        subgraph AUTO["Generated and managed by Foundry IQ: createdResources"]
            DS["Data source<br/>datasource: Blob connection and read definition"]
            IXR["Indexer<br/>indexer: runs the ingestion"]
            SK["Skillset<br/>skillset: chunking, vectorization, triple extraction"]
            CI[("Content index<br/>index: text chunks + vectors")]
            GI[("Graph index<br/>graphIndex: triples + evidence chunk IDs")]
            DS --> IXR --> SK
            SK -->|T5 and T6: store chunks| CI
            SK -->|T8: store extracted edges| GI
            GI -. referenced by evidence chunk ID .-> CI
        end
        KB["Knowledge base: ft-entity-kb (optional)<br/>T13: created separately when RUN_KB_TEST = True"]
        KS -. generates and configures per the definition .-> AUTO
        KB -. references the knowledge source .-> KS
    end

    subgraph AOAI["Azure OpenAI (existing model deployments)"]
        EMB["Embedding model<br/>vectorization"]
        CHAT["Chat model<br/>triple extraction / GQL and answer generation"]
    end
    subgraph CACHE["Cache storage (existing account)"]
        EC[("Cache generated automatically by Foundry IQ<br/>Blob container + table")]
    end

    BLOB -->|read the articles| DS
    IXR <-->|save and reuse processing results| EC
    SK -. request vectorization .-> EMB
    SK -. T8: request triple extraction .-> CHAT
    KB -. request GQL and answer generation .-> CHAT

    classDef notebook fill:#dbeafe,stroke:#2563eb,color:#172554
    classDef generated fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef existing fill:#f1f5f9,stroke:#64748b,color:#0f172a
    class BLOB,KS,KB notebook
    class DS,IXR,SK,CI,GI,EC generated
    class EMB,CHAT existing
    style KB stroke-dasharray:5 5
```

- **Check the generated names:** In the diagram, `datasource`, `indexer`, `skillset`, `index` and `graphIndex` are keys of `azureBlobParameters.createdResources`. Get the actual resource names from the service response. `graphIndex` may be missing from the response right after creation, so the notebook fetches the definition again after the T8 sync completes.
- **Ontology and cache:** The ontology is JSON-LD embedded in the knowledge source's `graphExtraction.ontology`, and it is not a separate Foundry IQ resource. T5 and T6 do the chunking and vectorization, and T8 reuses the cache to extract triples. The graph name `ft-entity` is the value of `graphExtraction.graphName`, which is distinct from the resource name of the graph index.
- **Not created automatically:** The Foundry IQ service, storage accounts and model deployments are existing ones. The knowledge base is also not generated automatically when you create the knowledge source. The notebook creates it separately only when you enable T13.

> Source: [Objects that a Blob knowledge source generates automatically](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob) / [Resources generated by an Entity Graph, and the knowledge base](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#create-the-graph-enabled-knowledge-source) / [Blob container and table of the enrichment cache](https://learn.microsoft.com/azure/search/enrichment-cache-how-to-configure) / [Attaching an ontology later and reusing the cache](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md)

## Test flow

```mermaid
flowchart LR
    TXT["2 text files"] --> BLOB["Upload to Blob"] --> KS["Create the knowledge source<br/>graphExtraction + cache<br/>no ontology"]
    KS --> SYNC["Sync<br/>chunking + vectorization"] --> CI[("Content index")]
    SYNC --> ATT["Attach the ontology"] --> TRI["Extract triples<br/>reuse the cache"] --> GI[("Graph index")]
    GI --> Q["GQL / provenance / expected facts"]
    CI --> Q
    GI --> VIS["Visualize the registered ontology and entities"] --> EXPLORE["GQL, aggregation, evidence text"]
```

## Test list

| ID | Target | Pass condition |
| --- | --- | --- |
| T1 | Connection | The SDK is 12.2.0b1. You can connect to Foundry IQ. `SearchClient.graph` exists |
| T2 | Input | The two files exist, and no tags or `\n` strings remain after cleaning |
| T3 | Storage | For the input source: hierarchical namespace disabled, Blob soft delete enabled, versioning disabled. The cache destination is general-purpose v2 |
| T4 | Upload | The two blobs are placed |
| T5 | Create the knowledge source (no ontology) | The graph name is set, the ontology is empty, and the content index and indexer are generated automatically |
| T6 | First sync | 0 failures. Chunks are created. **The graph has no edges yet** |
| T7 | Ontology | At most 128 declarations and at most 65,536 UTF-16 code units |
| T8 | Attach the ontology | Triples are written. The chunk count does not change (cache reuse). The ontology remains in the definition |
| T9 | Graph index | `edgeList` profile. Edges exist, and every edge has an evidence chunk attached |
| T10 | Expected facts | Recall is at least `MIN_RECALL` |
| T11 | Nodes shared across documents | For each of location, event and designation, the two heritage sites connect to the same node |
| T12 | Multi-hop and provenance | The 3-hop result includes the founder, and the text of the evidence chunk can be retrieved |
| T13 | Knowledge base (optional) | The answer comes with references and an execution log of the graph search |
| T14 | Visualize registered contents | You can retrieve the registered ontology, the relationships between entities and the evidence chunks |

## Prerequisites

### Required resources

| Resource | Purpose |
| --- | --- |
| Foundry IQ (in a region where you can create knowledge sources and knowledge bases) | Knowledge sources, indexes |
| Azure OpenAI (chat model and embedding model deployments) | Graph extraction, vectorization |
| Blob Storage (for the source, hierarchical namespace disabled) | Holds the article text. Enable Blob soft delete and disable versioning before the first ingestion |
| General-purpose v2 storage account (for the cache, `StorageV2`) | Blob container and table for the enrichment cache |

The official Entity Graph procedure **does not support ADLS Gen2 as an input source**. It also requires you to enable native Blob soft delete and disable Blob versioning before the first ingestion. T3 checks these settings and, if they do not comply, stops the deletion, upload and knowledge source creation.

The official requirement for the cache destination is general-purpose v2. The cited official documents do not confirm a condition that the cache destination must have the hierarchical namespace disabled, so this notebook does not conflate it with the input-source restriction. Check any error the service returns at run time as it is.

> Source: [Entity Graph prerequisites](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#prerequisites) / [Enrichment cache prerequisites](https://learn.microsoft.com/azure/search/enrichment-cache-how-to-configure#prerequisites)

### Roles

| Assigned to | Role | Scope |
| --- | --- | --- |
| The user who runs the notebook | Search Service Contributor, Search Index Data Contributor | Foundry IQ service |
| The user who runs the notebook | Storage Blob Data Contributor | Source storage |
| Foundry IQ managed identity | Storage Blob Data Reader | Source storage |
| Foundry IQ managed identity | Storage Blob Data Contributor, Storage Table Data Contributor | Cache storage |
| Foundry IQ managed identity | Cognitive Services OpenAI User | Azure OpenAI |

### `.env`

- **Reusing an existing graph:** Only `AZURE_SEARCH_ENDPOINT` is required. You do not need the input files, storage administration permissions or the embedding model settings. The resource and role lists above are the prerequisites for a new build. Reuse requires permission to read the knowledge source and index definitions and to search the index.
- **New build:** You additionally need `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_STORAGE_RESOURCE_ID` and `AZURE_CACHE_STORAGE_RESOURCE_ID` (and, optionally, `AZURE_OPENAI_MODEL` and `AZURE_OPENAI_EMBEDDING_MODEL`).
- **Running T13:** Even in reuse mode, you need `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_DEPLOYMENT`. You also need permission to create and update the knowledge base.

> Source: [Create a Blob Knowledge Source for Agentic Retrieval](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob) / [Connect existing resources(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-provision/references/connect-existing-resources.md)

## 1. Setup

### 1.1 Install the packages

The knowledge graph models and APIs (`GraphProfile`, `KnowledgeSourceGraphExtractionParameters`, `SearchClient.graph` and so on) were added in `azure-search-documents` 12.2.0b1. The version is pinned, so `--pre` is not needed.

> Source: [azure-search-documents CHANGELOG](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md)

In [ ]:
%pip install -q azure-search-documents==12.2.0b1 azure-identity azure-storage-blob python-dotenv pandas networkx matplotlib requests

### 1.2 Settings

The notebook loads the endpoints from `.env`. Authentication uses Microsoft Entra ID (`DefaultAzureCredential`). `check()` records each test result in `results`. As a rule, failures are also displayed together at the end, but the notebook stops with an exception if the T3 storage requirements are not met. T4 and T5 do not write anything unless T3 succeeded for the current input source and cache destination.

| Name | Meaning |
| --- | --- |
| `RUN_BUILD` | Defaults to `True`. Runs T2 to T8, and T4 deletes and recreates resources with the same names. Set it to `False` to only search an existing graph |
| `KS_NAME` | Defaults to `ft-entity-ks`. Change it to use a different existing graph |
| `ONTOLOGY_PATH` | The default `None` uses the built-in ontology. Specify a path only when you use an external, validated JSON-LD |
| `MAX_DISPLAY_EDGES` | The upper limit on the number of documents to retrieve when drawing the registered graph |
| `MIN_RECALL` | The lower limit on the share of the "expected facts" found in the graph. LLM extraction varies, so 100% is not required |
| `RUN_KB_TEST` | When `True`, the knowledge base functional test (T13) also runs. This incurs additional model costs |
| `RUN_CLEANUP` | When `True`, deletes the objects created in new-build mode at the end. It cannot delete anything in reuse mode |

In [ ]:
import hashlib
import json
import logging
import os
import re
import time
from importlib.metadata import version
from pathlib import Path

from IPython.display import display

import pandas as pd
import requests
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True)
credential = DefaultAzureCredential()

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
API_VERSION = "2026-11-01-preview"
RUN_BUILD = False
RUN_KB_TEST = False
RUN_CLEANUP = False
VALIDATED_STORAGE_IDS = None

FT = "ft-entity"  # prefix for the names of the objects this test creates
GRAPH_NAME = FT
KS_NAME = f"{FT}-ks"
KB_NAME = f"{FT}-kb"
CONTAINER = FT
MIN_RECALL = 0.6
ONTOLOGY_PATH = None  # set Path(...) only when you use an external JSON-LD
MAX_DISPLAY_EDGES = 200
if not 1 <= MAX_DISPLAY_EDGES <= 1000:
    raise ValueError("Set MAX_DISPLAY_EDGES to a value from 1 to 1000.")

if RUN_BUILD or RUN_KB_TEST:
    AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
    CHAT_MODEL = os.environ.get("AZURE_OPENAI_MODEL", CHAT_DEPLOYMENT)
if RUN_BUILD:
    EMBED_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
    EMBED_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_MODEL", EMBED_DEPLOYMENT)
    SOURCE_STORAGE_ID = os.environ["AZURE_STORAGE_RESOURCE_ID"]
    CACHE_STORAGE_ID = os.environ["AZURE_CACHE_STORAGE_RESOURCE_ID"]

results = []


def check(case, ok, detail=""):
    results.append({"Test": case, "Result": "PASS" if ok else "FAIL", "Detail": str(detail)[:200]})
    print("PASS" if ok else "FAIL", "-", case, f"| {detail}" if detail else "")
    return ok


print("Run mode:", "New build (T4 deletes resources with the same names)" if RUN_BUILD else "Reuse an existing graph (T2 to T8 are skipped)")
pd.DataFrame({"Value": [SEARCH_ENDPOINT, KS_NAME]}, index=["Foundry IQ", "Knowledge source"])

### 1.3 Test schema

These are the classes, relations and attributes shared by the extraction in a new build and the per-relation searches in T10. Run this cell in reuse mode too. It only defines local tables and does not change any existing ontology or graph. The assembly into JSON-LD and the attachment happen in T7 and T8 of new-build mode.

> Source: [Entity Graph ontology](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph)

In [ ]:
classes = pd.DataFrame([
    ("HeritageSite", "A heritage site such as a castle, park or historic building. Examples: Elowen Park, Thornwick Castle"),
    ("Feature", "A signature object, artwork or natural feature of a site. Examples: Twelve Stars Tapestry, Elowen Cherry"),
    ("Person", "A person. Examples: Osric Thorne, Queen Elizabeth I"),
    ("Facility", "A building, garden or room within the grounds. Examples: Great Hall, Pavilion, Bindery, Moonwell Garden"),
    ("Place", "A place name or administrative area. Examples: Eastgate Ward, Aldermere"),
    ("Event", "A festival, ceremony or historical event. Examples: Blossom Festival, Great Fire of Aldermere"),
    ("Designation", "A designation or inscription such as World Heritage status. Examples: Historic Monuments of Old Aldermere"),
], columns=["class", "comment"])

relations = pd.DataFrame([
    ("HeritageSite", "locatedIn", "Place", "The area where the site is located"),
    ("HeritageSite", "hasFeature", "Feature", "A signature object or feature of the site"),
    ("HeritageSite", "foundedBy", "Person", "The person who founded the site"),
    ("HeritageSite", "hasFacility", "Facility", "A building or garden within the grounds"),
    ("HeritageSite", "survivedEvent", "Event", "An event, such as a fire, that the site escaped"),
    ("HeritageSite", "registeredAs", "Designation", "A designation under which the site is registered"),
    ("Person", "visited", "Facility", "A building or garden that the person visited"),
], columns=["domain", "predicate", "range", "comment"])

datatype_properties = pd.DataFrame([("HeritageSite", "foundedYear", "integer", "The calendar year in which the site was founded")], columns=["domain", "predicate", "xsd_range", "comment"])
relations

### 1.4 Search helpers

This section defines the GQL execution, the conversion of results into tables and the normalization of spelling variants in names, all used in T10 to T12. The SDK models that retrieve provenance are also imported here. The search target and the graph-name filter are set in T9. Defining the helpers does not run any search.

> Source: [Documents - Graph](https://learn.microsoft.com/en-us/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
from azure.search.documents.models import GraphQueryProvenanceHydrateDocumentsParameters, GraphQueryProvenanceParameters


def run_gql(gql, provenance=None):
    result = graph_client.graph(gql=gql, filter=GRAPH_FILTER, include_statistics=True, provenance=provenance)
    print(gql, "\n   rows:", result.count)
    return result


def rows(result):
    return pd.DataFrame([r.as_dict() for r in result.value]).drop(columns=["@search.graphProvenance"], errors="ignore")


def norm(s):
    return re.sub(r"[\W_]+", "", s or "").casefold()

### 1.5 Drawing helpers

These drawing routines are shared by the ontology, the whole searched graph and the multi-hop paths. Run this cell whether or not you do a new build.

To draw multiple relations between the same nodes, the diagram of the registered contents uses a `MultiDiGraph`.

> Source: [NetworkX graph drawing](https://networkx.org/documentation/stable/reference/drawing.html)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
matplotlib.rcParams["font.sans-serif"] = ["Segoe UI", "Arial", "DejaVu Sans"]


import textwrap

def draw_graph(graph, title, node_size=1400, figsize=(13, 9), positions=None):
    if not graph:
        raise ValueError("There are no relationships to draw. Check the number of retrieved items and the ontology.")
    kinds = sorted({data.get("kind", data.get("type", "?")) for _, data in graph.nodes(data=True)})
    palette = {kind: plt.cm.tab20(number % 20) for number, kind in enumerate(kinds)}
    if positions is None:
        positions = nx.spring_layout(graph, seed=7, k=1.5)
    styles = [f"arc3,rad={number * 0.12}" for number in range(10)]
    fig, ax = plt.subplots(figsize=figsize)
    nx.draw_networkx_nodes(graph, positions, ax=ax, node_size=node_size, alpha=0.9,
                           node_color=[palette[graph.nodes[node].get("kind", graph.nodes[node].get("type", "?"))] for node in graph])
    nx.draw_networkx_edges(graph, positions, ax=ax, arrows=True, arrowsize=15, node_size=node_size, connectionstyle=styles, edge_color="#777777")
    nx.draw_networkx_labels(graph, positions, ax=ax, font_size=9,
                            labels={node: textwrap.fill(str(data.get("label", node)), 15) for node, data in graph.nodes(data=True)},
                            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.5})
    nx.draw_networkx_edge_labels(graph, positions, ax=ax, edge_labels=nx.get_edge_attributes(graph, "predicate"),
                                 font_size=7, rotate=False, node_size=node_size, connectionstyle=styles)
    ax.legend(handles=[Patch(color=palette[kind], label=kind) for kind in kinds], loc="upper left")
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### 1.6 Validate an external JSON-LD (optional)

If `ONTOLOGY_PATH = None`, the following validation and conversion cells do not run, and the runtime is not needed. If you specify an external file, first fetch the official `foundry-iq-skills` by following [the manual download and placement steps in the README](./README.md) (section 5.2). This distribution does not bundle it and does not download it automatically. If you place it in another location, specify the location with `FOUNDRY_IQ_PLUGIN_ROOT`. The notebook does not modify the input file, and it uses the content validated and converted in the `RUN_BUILD = True` creation path.

> Source: [Ontology conversion code](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

To detect problems in an external file before any resource deletion, creation or model call, the setup validates the file first when you specify one.

In [ ]:
if RUN_BUILD and ONTOLOGY_PATH is not None:
    import subprocess
    import sys

    ONTOLOGY_PATH = Path(ONTOLOGY_PATH).expanduser().resolve()
    plugin_root = Path(os.environ.get("FOUNDRY_IQ_PLUGIN_ROOT", "foundry-iq-skills")).expanduser().resolve()
    BOOTSTRAP = plugin_root / "graph" / "runtime" / "bootstrap.py"
    if not BOOTSTRAP.is_file():
        raise FileNotFoundError(f"The Foundry IQ runtime is not in place: {BOOTSTRAP}. Follow the manual download and placement steps in the README and put foundry-iq-skills in the same folder as the notebook, or set FOUNDRY_IQ_PLUGIN_ROOT to its location. The notebook does not download it automatically.")
    print("Validation runtime:", BOOTSTRAP)

In [ ]:
if RUN_BUILD and ONTOLOGY_PATH is not None:
    if not ONTOLOGY_PATH.is_file():
        raise FileNotFoundError(f"{ONTOLOGY_PATH} does not exist. Check ONTOLOGY_PATH. Set it to None to use the built-in ontology.")
    if not BOOTSTRAP.is_file():
        raise FileNotFoundError(f"The runtime does not exist: {BOOTSTRAP.resolve()}. Re-run the settings cell and check the validation runtime it prints.")
    validation = subprocess.run([sys.executable, "-I", str(BOOTSTRAP.resolve()), "run", "ontology", "ontology-consume", "check", str(ONTOLOGY_PATH.resolve()), "--json"],
                                capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120)
    print(validation.stdout, validation.stderr)
    validation.check_returncode()
    profile = json.loads(validation.stdout)
    if profile["status"] != "ok" or profile["violations"]:
        raise ValueError("The ontology profile validation did not pass.")
    ontology_text = ONTOLOGY_PATH.read_text(encoding="utf-8")
    ontology_hash = hashlib.sha256(ontology_text.encode("utf-8")).hexdigest()
    print("Ontology SHA-256:", ontology_hash)

### 1.7 Create the JSON-LD to send to Foundry IQ from the external JSON-LD

Identifiers in the generated file such as `urn:ontology:...#Castle` may be rejected by Foundry IQ even though they are valid JSON-LD. The notebook uses `load_ontology(INLINE_JSONLD)` of the manually downloaded Entity Graph runtime to convert them into identifiers such as `entity:Castle`, which are defined in the embedded context. The namespace for registration is `https://schema.search.azure.com/graph-extraction#`.

The conversion carries over classes, relations, attributes, domain / range and the descriptions used for extraction. Management metadata such as the ontology header is not included in the definition that is sent. **The original file and the approved version are not modified.** The `ontology_hash` of the input file and the `search_ontology_hash` of what is actually sent are recorded separately. After registration, the notebook compares against the latter.

The conversion code is called from a wheel whose SHA-256 was checked against the manifest of the fetched runtime, in an isolated Python process. The child process also sets UTF-8 mode explicitly, which keeps non-ASCII text in descriptions intact. This processing uses only the standard library and performs no additional installation and no calls to Azure or models.

> Source: [Official Foundry IQ Skills distribution code (`ekg_core.ontology.load_ontology` / `_search_inline`)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels) / [Example of the embedded context for an Entity Graph](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#create-the-graph-enabled-knowledge-source)

In [ ]:
if RUN_BUILD and ONTOLOGY_PATH is not None:
    runtime_dir = BOOTSTRAP.resolve().parent
    runtime_manifest = json.loads((runtime_dir / "manifest.json").read_text(encoding="utf-8"))
    adapter_record = runtime_manifest["runtimes"]["entity"]["wheel"]
    adapter_wheel = (runtime_dir / adapter_record["file"]).resolve()
    if not adapter_wheel.is_relative_to(runtime_dir):
        raise ValueError("The conversion wheel points outside the runtime folder.")
    if hashlib.sha256(adapter_wheel.read_bytes()).hexdigest() != adapter_record["sha256"]:
        raise ValueError("The SHA-256 of the conversion wheel does not match the manifest.")

    import textwrap
    adapter_script = textwrap.dedent("""
    import json, sys
    sys.path.insert(0, sys.argv[1])
    from ekg_core.models import OntologyMode
    from ekg_core.ontology import load_ontology
    ontology = load_ontology(OntologyMode.INLINE_JSONLD, path=sys.argv[2])
    print(json.dumps({"source_sha256": ontology.schema_id, "inline_jsonld": ontology.inline_jsonld}, ensure_ascii=False))
    """)
    conversion = subprocess.run([sys.executable, "-I", "-X", "utf8", "-c", adapter_script, str(adapter_wheel), str(ONTOLOGY_PATH.resolve())],
                                capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120)
    if conversion.returncode:
        print(conversion.stdout, conversion.stderr)
    conversion.check_returncode()
    converted = json.loads(conversion.stdout)
    if converted["source_sha256"] != f"sha256:{ontology_hash}":
        raise RuntimeError("The input file changed after validation. Re-run from the original-file validation cell.")
    search_ontology_text = converted["inline_jsonld"]
    search_ontology_hash = hashlib.sha256(search_ontology_text.encode("utf-8")).hexdigest()
    display(pd.DataFrame([{"Format": "Original file (unchanged)", "SHA-256": ontology_hash},
                          {"Format": "JSON-LD sent to Foundry IQ", "SHA-256": search_ontology_hash}]))
    print("Context for sending to Foundry IQ:", json.loads(search_ontology_text)["@context"])

## 2. T1: Connection

This test checks the SDK version, the connection to Foundry IQ and the graph query API.

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient

index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, api_version=API_VERSION)
indexer_client = SearchIndexerClient(SEARCH_ENDPOINT, credential, api_version=API_VERSION)

existing = [ks.name for ks in index_client.list_knowledge_sources()]
check("T1-1 The SDK is 12.2.0b1", version("azure-search-documents") == "12.2.0b1", version("azure-search-documents"))
check("T1-2 Connect to Foundry IQ and list the knowledge sources", True, f"{len(existing)} existing knowledge sources")
check("T1-3 The graph query (SearchClient.graph) is available", hasattr(SearchClient, "graph"))

## 3. T2: Input text

`Thornwick Castle - Hanapedia.txt` and `Elowen Park - Hanapedia.txt` are articles from Hanapedia, a fictional free encyclopedia. The two articles share the following points, which you can use for questions that span documents.

| Shared point | Thornwick Castle | Elowen Park |
| --- | --- | --- |
| Location | Eastgate Ward, Aldermere | Eastgate Ward, Aldermere |
| World Heritage | Historic Monuments of Old Aldermere | Historic Monuments of Old Aldermere |
| Historical event | Escaped the Great Fire of Aldermere (1684) | Escaped the Great Fire of Aldermere (1684) |

The file extension is `.txt`, but the content is Markdown with headings, paragraphs and tables. Line breaks are saved as real line breaks, and there are no HTML comments with page information or escaped line breaks. The notebook runs the text through the existing cleaning step before uploading it. Blob names are ASCII.

In [ ]:
if RUN_BUILD:
    DATA_DIR = Path(".")  # same folder as the notebook
    SAMPLES = {"thornwick-castle.txt": "Thornwick Castle - Hanapedia.txt", "elowen-park.txt": "Elowen Park - Hanapedia.txt"}


def clean_text(raw):
    text = raw.replace("\\n", "\n").replace('\\"', '"')
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"<tr>(.*?)</tr>", lambda m: " / ".join(re.findall(r"<td>(.*?)</td>", m.group(1), flags=re.S)) + "\n", text, flags=re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


if RUN_BUILD:
    raw_texts = {blob: (DATA_DIR / src).read_text(encoding="utf-8") for blob, src in SAMPLES.items()}
    cleaned = {blob: clean_text(raw) for blob, raw in raw_texts.items()}

    check("T2-1 The 2 input files exist", all((DATA_DIR / src).is_file() for src in SAMPLES.values()), list(SAMPLES.values()))
    check("T2-2 After cleaning, no tags or `\\n` strings remain", all("<" not in t and "\\n" not in t for t in cleaned.values()))
    check("T2-3 Each cleaned text is longer than 1,000 characters", all(len(t) > 1000 for t in cleaned.values()), {b: len(t) for b, t in cleaned.items()})
    display(pd.DataFrame([{"Blob name": b, "Source file": SAMPLES[b], "Before cleaning": len(raw_texts[b]), "After cleaning": len(cleaned[b])} for b in SAMPLES]))

## 4. T3: Storage state

The notebook reads the settings of the input source and the cache destination from ARM and validates the requirements, which differ by purpose. It does not change any settings automatically.

| Purpose | Required conditions | Basis |
| --- | --- | --- |
| Input source | Hierarchical namespace disabled, native Blob soft delete enabled, Blob versioning disabled | Entity Graph prerequisites |
| Cache destination | General-purpose v2 storage account (`StorageV2`) | Enrichment cache prerequisites |

The hierarchical namespace, soft delete and versioning of the cache destination are only displayed, and the input-source conditions are not applied to them. In ARM, `isHnsEnabled` and `isVersioningEnabled` mean the setting is enabled when the value is `true`. If any required condition is not met, the notebook displays the results and then stops. If you fix a setting or change a storage resource ID, re-run T3.

The network settings are for reference. Being able to read the ARM settings does not confirm that Foundry IQ can reach the storage. An HTTP error is raised as an exception before the settings are read.

> Source: [Entity Graph prerequisites](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#prerequisites) / [Enrichment cache prerequisites](https://learn.microsoft.com/azure/search/enrichment-cache-how-to-configure#prerequisites) / [ARM settings of a storage account](https://learn.microsoft.com/rest/api/storagerp/storage-accounts/get-properties?view=rest-storagerp-2024-01-01) / [ARM settings of the Blob service](https://learn.microsoft.com/rest/api/storagerp/blob-services/get-service-properties?view=rest-storagerp-2024-01-01)

In [ ]:
if RUN_BUILD:
    VALIDATED_STORAGE_IDS = None
    arm = {"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"}
    storage_rows = []
    for label, rid in [("Source", SOURCE_STORAGE_ID), ("Cache", CACHE_STORAGE_ID)]:
        account_response = requests.get(f"https://management.azure.com{rid}?api-version=2024-01-01", headers=arm, timeout=30)
        account_response.raise_for_status()
        account_data = account_response.json()
        account = account_data["properties"]
        blob_response = requests.get(f"https://management.azure.com{rid}/blobServices/default?api-version=2024-01-01", headers=arm, timeout=30)
        blob_response.raise_for_status()
        blob = blob_response.json()["properties"]
        storage_rows.append({"Purpose": label, "Account": rid.split("/")[-1], "Account kind": account_data.get("kind"),
                             "Hierarchical namespace": bool(account.get("isHnsEnabled")),
                             "publicNetworkAccess": account.get("publicNetworkAccess"), "defaultAction": (account.get("networkAcls") or {}).get("defaultAction"),
                             "Blob soft delete": bool((blob.get("deleteRetentionPolicy") or {}).get("enabled")),
                             "Versioning": bool(blob.get("isVersioningEnabled"))})
    storage = pd.DataFrame(storage_rows)
    display(storage)
    source_storage = storage.loc[storage["Purpose"] == "Source"].iloc[0]
    cache_storage = storage.loc[storage["Purpose"] == "Cache"].iloc[0]
    storage_checks = [
        check("T3-1 The input source's hierarchical namespace is disabled", not source_storage["Hierarchical namespace"], source_storage["Hierarchical namespace"]),
        check("T3-2 Blob soft delete is enabled on the input source", bool(source_storage["Blob soft delete"]), source_storage["Blob soft delete"]),
        check("T3-3 Blob versioning is disabled on the input source", not source_storage["Versioning"], source_storage["Versioning"]),
        check("T3-4 The cache destination is general-purpose v2", cache_storage["Account kind"] == "StorageV2", cache_storage["Account kind"]),
    ]
    if not all(storage_checks):
        raise RuntimeError("The T3 storage requirements are not met. Fix the settings marked FAIL and re-run T3. The notebook does not run deletion, upload or knowledge source creation.")
    VALIDATED_STORAGE_IDS = (SOURCE_STORAGE_ID, CACHE_STORAGE_ID)

## 5. T4: Clean up the previous run and upload to Blob

If a previous run remains, the notebook deletes the knowledge base and knowledge source whose names start with `ft-entity`. Deleting the knowledge source also deletes the automatically generated data source, skillset, indexer and index. After the deletion takes effect, the notebook places the cleaned text in the Blob container.

In [ ]:
from azure.storage.blob import BlobServiceClient, ContentSettings

if RUN_BUILD:
    if VALIDATED_STORAGE_IDS != (SOURCE_STORAGE_ID, CACHE_STORAGE_ID):
        raise RuntimeError("Run the T3 validation successfully for the current storage first. The T4 deletion and upload do not run.")
    for delete, name in ((index_client.delete_knowledge_base, KB_NAME), (index_client.delete_knowledge_source, KS_NAME)):
        try:
            delete(name)
        except ResourceNotFoundError:
            pass
    for _ in range(60):  # wait for the deletion to take effect
        try:
            index_client.get_knowledge_source(KS_NAME)
            time.sleep(5)
        except ResourceNotFoundError:
            break

    blob_service = BlobServiceClient(f"https://{SOURCE_STORAGE_ID.split('/')[-1]}.blob.core.windows.net", credential)
    container = blob_service.get_container_client(CONTAINER)
    if not container.exists():
        container.create_container()
    for blob_name, text in cleaned.items():
        container.upload_blob(blob_name, text.encode("utf-8"), overwrite=True, content_settings=ContentSettings(content_type="text/plain; charset=utf-8"))

    blobs = [b.name for b in container.list_blobs()]
    check("T4 The 2 blobs are placed", set(blobs) == set(SAMPLES), blobs)

## 6. T5: Create the knowledge source (no ontology)

When you create an Azure Blob knowledge source, Foundry IQ generates the following objects automatically (a data source, a skillset, a content index and an indexer). Add the following settings.

| Setting | Meaning |
| --- | --- |
| `is_adls_gen2=False` | Specifies the Entity Graph input as standard Azure Blob Storage. T3 validates the actual hierarchical namespace setting |
| `graph_extraction=KnowledgeSourceGraphExtractionParameters(graph_name=...)` | Enables graph extraction and names the graph. **No ontology is attached at this point** |
| `cache=KnowledgeSourceCache(..., enable_reprocessing=True)` | Enables the enrichment cache. When you attach the ontology, the service reprocesses from the cache without redoing the chunking and vectorization |
| `content_extraction_mode="minimal"`, `disable_image_verbalization=True` | Handle text only |

The first sync starts right after you call the create operation.

> Source: [Create a Blob Knowledge Source for Agentic Retrieval](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob) / [SearchServiceCreateKnowledgeSourceAzureBlobGraphCache.json(REST specification example)](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceCreateKnowledgeSourceAzureBlobGraphCache.json) / [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md)

### Check the capacity of the destination

The notebook expects 2 indexes (for the content and the graph) and 1 each of knowledge source, data source, indexer and skillset. Checking the capacity and actually creating the resources are separate operations.

> Source: [Service statistics](https://learn.microsoft.com/rest/api/searchservice/get-service-statistics/get-service-statistics?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_BUILD:
    counters = index_client.get_service_statistics().counters
    requirements = [("Index", counters.index_counter, 2),
                    ("Knowledge source", counters.knowledge_source_counter, 1),
                    ("Data source", counters.data_source_counter, 1),
                    ("Indexer", counters.indexer_counter, 1),
                    ("Skillset", counters.skillset_counter, 1)]
    if any(counter is None or counter.quota is None for _, counter, _ in requirements):
        raise RuntimeError("Cannot confirm the required capacity limits. Ask an administrator before writing.")
    capacity = pd.DataFrame([{"Target": name, "In use": counter.usage, "Limit": counter.quota, "Needed now": needed}
                             for name, counter, needed in requirements])
    display(capacity)
    if ((capacity["Limit"] - capacity["In use"]) < capacity["Needed now"]).any():
        raise RuntimeError("There is not enough capacity for the new registration. Existing resources are not deleted automatically.")

In [ ]:
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSource,
    AzureBlobKnowledgeSourceParameters,
    AzureOpenAIVectorizerParameters,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceGraphExtractionParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceCache,
    KnowledgeSourceIngestionParameters,
)

if RUN_BUILD:
    if VALIDATED_STORAGE_IDS != (SOURCE_STORAGE_ID, CACHE_STORAGE_ID):
        raise RuntimeError("Run the T3 validation successfully for the current storage first. The knowledge source creation does not run.")
    embedding_params = AzureOpenAIVectorizerParameters(resource_url=AOAI_ENDPOINT, deployment_name=EMBED_DEPLOYMENT, model_name=EMBED_MODEL)
    chat_params = AzureOpenAIVectorizerParameters(resource_url=AOAI_ENDPOINT, deployment_name=CHAT_DEPLOYMENT, model_name=CHAT_MODEL)

    knowledge_source = AzureBlobKnowledgeSource(
        name=KS_NAME,
        description="Hanapedia articles about the heritage sites of Aldermere (Thornwick Castle and Elowen Park). They cover history, signature features, facilities in the grounds, heritage assets and related people.",
        azure_blob_parameters=AzureBlobKnowledgeSourceParameters(
            connection_string=f"ResourceId={SOURCE_STORAGE_ID}",
            container_name=CONTAINER,
            is_adls_gen2=False,
            ingestion_parameters=KnowledgeSourceIngestionParameters(
                embedding_model=KnowledgeSourceAzureOpenAIVectorizer(azure_open_ai_parameters=embedding_params),
                chat_completion_model=KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=chat_params),
                disable_image_verbalization=True,
                content_extraction_mode="minimal",
                cache=KnowledgeSourceCache(storage_connection_string=f"ResourceId={CACHE_STORAGE_ID}", enable_reprocessing=True),
            ),
            graph_extraction=KnowledgeSourceGraphExtractionParameters(graph_name=GRAPH_NAME),
        ),
    )

    created = index_client.create_or_update_knowledge_source(knowledge_source)
    resources = created.azure_blob_parameters.created_resources
    graph_extraction = created.azure_blob_parameters.graph_extraction

    check("T5-1 The graph name is set and the ontology is empty", graph_extraction.graph_name == GRAPH_NAME and graph_extraction.ontology is None, f"graph_name={graph_extraction.graph_name} ontology={graph_extraction.ontology}")
    check("T5-2 The content index and indexer are generated automatically", bool(resources.get("index")) and bool(resources.get("indexer")), {k: v for k, v in resources.items() if v})
    display(pd.DataFrame({"Auto-generated objects": {k: v for k, v in resources.items() if v}}).rename_axis("Type"))

## 7. T6: First sync (chunking and vectorization only)

`get_knowledge_source_status` returns the sync in progress (`current_synchronization_state`) and the most recently completed sync (`last_synchronization_state`). If the in-progress value is `None` and a most recently completed sync exists, the sync is finished.

The knowledge source status does not include error messages, so the notebook also displays the latest run of the generated indexer. When `transientFailure` continues, this error message shows the cause (reachability to the storage, missing roles and so on).

> Source: [KnowledgeSourceStatus(azure-search-documents 12.2.0b1)](https://pypi.org/project/azure-search-documents/12.2.0b1/) / [Status while a step runs(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/status-and-telemetry.md)

In [ ]:
def wait_sync(done, timeout=1800, interval=30, max_error_polls=10, rerun=False):
    started, error_polls, reruns = time.time(), 0, set()
    while time.time() - started < timeout:
        status = index_client.get_knowledge_source_status(KS_NAME)
        current, last = status.current_synchronization_state, status.last_synchronization_state
        run = indexer_client.get_indexer_status(resources["indexer"]).last_result
        shown = current or last
        print(f"{int(time.time() - started):>5}s status={status.synchronization_status} running={'yes' if current else 'no'} "
              f"processed={getattr(shown, 'items_updates_processed', '-')} failed={getattr(shown, 'items_updates_failed', '-')} "
              f"indexer={getattr(run.status, 'value', run.status) if run else '-'}")
        indexer_error = run.error_message if run else None
        if indexer_error:
            print("   Indexer error:", indexer_error[:300])
        for error in (run.errors if run else None) or []:
            print("   Document error:", (error.error_message or "")[:300])
        if done(current, last):
            return status
        if rerun and indexer_error and "must be retried" in indexer_error.lower() and run.start_time not in reruns and len(reruns) < 5:
            reruns.add(run.start_time)  # documents are still waiting to be processed, so the service asks for a rerun
            try:
                indexer_client.run_indexer(resources["indexer"])
            except HttpResponseError as e:
                if e.status_code != 409:
                    raise
            print("   Re-ran the indexer as the service requested")
            error_polls = 0
        else:
            error_polls = error_polls + 1 if indexer_error else 0
        if error_polls >= max_error_polls:
            raise RuntimeError("The indexer keeps returning errors, so the run is aborted. Things to check: "
                               "(1) whether the cache storage is a standard Blob account with the hierarchical namespace (ADLS Gen2) disabled, "
                               "(2) whether the Foundry IQ managed identity has the storage roles (including Storage Table Data Contributor for the cache), "
                               "(3) whether the storage allows the trusted Microsoft services exception (bypass=AzureServices) so that Foundry IQ can reach it.\n" + indexer_error[:300])
        time.sleep(interval)
    raise TimeoutError("The sync did not finish in time")


if RUN_BUILD:
    status = wait_sync(lambda current, last: current is None and last is not None)
    last = status.last_synchronization_state
    display(pd.DataFrame([{"Start": last.start_time, "End": last.end_time, "Processed": last.items_updates_processed, "Failed": last.items_updates_failed}]))

### 7.1 State of the content index and the graph index

The generated content index holds the chunks (`snippet`) and their vectors (`snippet_vector`). The evidence that a graph edge points to is a document in this index. **While there is no ontology, triple extraction does not run**, so the graph index should not exist yet (or, if it does, it should have 0 edges).

In [ ]:
if RUN_BUILD:
    content_index = index_client.get_index(resources["index"])
    content_client = SearchClient(SEARCH_ENDPOINT, content_index.name, credential)
    chunks_before = content_client.get_document_count()

    graph_index_before = index_client.get_knowledge_source(KS_NAME).azure_blob_parameters.created_resources.get("graphIndex")
    edges_before = SearchClient(SEARCH_ENDPOINT, graph_index_before, credential).get_document_count() if graph_index_before else 0

    check("T6-1 There are 0 sync failures", last.items_updates_failed == 0, f"Processed {last.items_updates_processed} / Failed {last.items_updates_failed}")
    check("T6-2 Chunks are created", chunks_before >= 2, f"{chunks_before} items")
    check("T6-3 While there is no ontology, the graph has no edges", edges_before == 0, f"graphIndex={graph_index_before} / edges {edges_before}")
    display(pd.DataFrame(list(content_client.search("*", select=["uid", "snippet_parent_id", "blob_url", "snippet"], top=3))))

## 8. T7: Define the ontology

An **ontology** is the blueprint of the entity types (classes) to put in the graph and the relationships (properties) between them. Following it, the LLM for extraction pulls subjects, predicates and objects out of the chunks. The descriptions and examples in `rdfs:comment` become the extraction instructions as they are.

The format is JSON-LD (OWL vocabulary). The service validates it when you attach it and rejects invalid ontologies. The main constraints are:

- The top level has only `@context` and `@graph`. `@type` is a single string.
- At most 128 declarations (classes and properties), and at most 65,536 UTF-16 code units in total.
- Each property has exactly one `rdfs:domain` and one `rdfs:range`.
- The range of a datatype property is one of `xsd:string`, `xsd:double`, `xsd:decimal`, `xsd:integer`, `xsd:boolean`, `xsd:date` and `xsd:dateTime`.

By default, the notebook uses a small ontology of 7 classes, 7 relations and 1 attribute tailored to the two articles. If you specify an external JSON-LD, the notebook selects the content validated and converted in 1.6 and 1.7. The smaller you keep the ontology, the more stable the extraction becomes and the lower the cost.

> Source: [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md) / [SearchServiceAttachKnowledgeSourceGraphOntology.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceAttachKnowledgeSourceGraphOntology.json) / [JSON-LD 1.1(W3C)](https://www.w3.org/TR/json-ld11/) / [OWL 2 Web Ontology Language Document Overview(W3C)](https://www.w3.org/TR/owl2-overview/)

### 8.1 Fix the ontology used for extraction

The notebook selects the built-in definition or the validated and converted external JSON-LD, and checks the number of declarations and the size.

In [ ]:
if RUN_BUILD:
    graph_nodes = [{"@id": f"entity:{r['class']}", "@type": "owl:Class", "rdfs:comment": r["comment"]} for _, r in classes.iterrows()]
    graph_nodes += [{"@id": f"entity:{r.predicate}", "@type": "owl:ObjectProperty", "rdfs:domain": {"@id": f"entity:{r.domain}"},
                     "rdfs:range": {"@id": f"entity:{r.range}"}, "rdfs:comment": r.comment} for r in relations.itertuples()]
    graph_nodes += [{"@id": f"entity:{r.predicate}", "@type": "owl:DatatypeProperty", "rdfs:domain": {"@id": f"entity:{r.domain}"},
                     "rdfs:range": {"@id": f"xsd:{r.xsd_range}"}, "rdfs:comment": r.comment} for r in datatype_properties.itertuples()]

    ontology = {
        "@context": {"owl": "http://www.w3.org/2002/07/owl#", "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
                     "xsd": "http://www.w3.org/2001/XMLSchema#", "entity": "https://schema.search.azure.com/graph-extraction#"},
        "@graph": graph_nodes,
    }
    ontology_jsonld = json.dumps(ontology, ensure_ascii=False, separators=(",", ":"))
    if ONTOLOGY_PATH is not None:
        ontology_jsonld = search_ontology_text
        ontology = json.loads(ontology_jsonld)
        graph_nodes = ontology["@graph"]
    search_ontology_hash = hashlib.sha256(ontology_jsonld.encode("utf-8")).hexdigest()
    units = len(ontology_jsonld.encode("utf-16-le")) // 2
    if len(graph_nodes) > 128 or units > 65536:
        raise ValueError("The number or size of the ontology declarations exceeds the limit.")

    check("T7-1 At most 128 declarations", len(graph_nodes) <= 128, f"{len(graph_nodes)} declarations")
    check("T7-2 At most 65,536 UTF-16 code units", units <= 65536, f"{units}")
    display(pd.DataFrame(graph_nodes))

### 8.2 Draw the ontology

In new-build mode, the notebook draws the classes of the test schema as nodes and the relations as arrows. It uses `draw_graph` defined in the common setup (node colors are per kind).

In [ ]:
if RUN_BUILD:
    ontology_graph = nx.MultiDiGraph()
    for declaration in ontology["@graph"]:
        if declaration["@type"] == "owl:Class":
            ontology_graph.add_node(declaration["@id"], label=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]), kind="Class")
    for declaration in ontology["@graph"]:
        if declaration["@type"] == "owl:ObjectProperty":
            ontology_graph.add_edge(declaration["rdfs:domain"]["@id"], declaration["rdfs:range"]["@id"],
                                    predicate=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]))
    draw_graph(ontology_graph, "Ontology registered in Foundry IQ: classes and relations")

## 9. T8: Attach the ontology and extract triples

This test runs in new-build mode only. The notebook adds the ontology (`kind="inline"`) to the knowledge source definition and updates it under the same name. It adds only the ontology to the `knowledge_source` object assembled in T5 and sends it again. The definition fetched from the service comes back with the cache's storage connection string hidden, so resending it as is would be rejected with an error whose gist is that the storage binding of the enrichment cache cannot be changed. Reuse mode does not update the definition.

It has been confirmed in a real environment that attaching alone does not start the extraction (the plugin also runs the indexer explicitly after attaching). The notebook therefore runs the generated indexer immediately after attaching. If the indexer is already running, the service returns 409, which the notebook ignores. When the indexer runs, only the triple extraction that **reuses the cache** runs, and edges are written to the graph index. **An attached ontology cannot be removed.**

The completion check is the same as in the plugin. The run is complete when no sync is in progress and the end time of the most recent sync is later than it was before the attachment. The indexer may finish with `must be retried`, and a rerun recovers from it (`rerun=True`, up to 5 times). Check the number of written triples with `graphStatistics.triplesWritten`.

> Source: [SearchServiceAttachKnowledgeSourceGraphOntology.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceAttachKnowledgeSourceGraphOntology.json) / [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md)

In [ ]:
from azure.search.documents.indexes.models import KnowledgeSourceGraphOntology

if RUN_BUILD:
    before_attach = index_client.get_knowledge_source_status(KS_NAME).last_synchronization_state.end_time

    knowledge_source.azure_blob_parameters.graph_extraction.ontology = KnowledgeSourceGraphOntology(kind="inline", name=f"{GRAPH_NAME}-ontology", value=ontology_jsonld)
    attached = index_client.create_or_update_knowledge_source(knowledge_source)
    resources = attached.azure_blob_parameters.created_resources

    try:
        indexer_client.run_indexer(resources["indexer"])  # start the triple extraction
    except HttpResponseError as e:
        if e.status_code != 409:  # 409 means it is already running
            raise

    status = wait_sync(lambda current, last: current is None and last is not None and last.end_time > before_attach, rerun=True)
    state = status.as_dict()
    graph_stats = state.get("graphStatistics") or state["lastSynchronizationState"].get("graphStatistics") or {}
    chunks_after = content_client.get_document_count()
    live = index_client.get_knowledge_source(KS_NAME).azure_blob_parameters.graph_extraction

    check("T8-1 Triples are written", graph_stats.get("triplesWritten", 0) > 0, graph_stats)
    check("T8-2 The chunk count does not change (chunking and vectorization were not redone)", chunks_after == chunks_before, f"Before attaching {chunks_before} / After attaching {chunks_after}")
    check("T8-3 The ontology remains in the knowledge source definition", live.ontology is not None and live.ontology.kind == "inline", getattr(live.ontology, "kind", None))
    registered_ontology_hash = hashlib.sha256(live.ontology.value.encode("utf-8")).hexdigest()
    if registered_ontology_hash != search_ontology_hash:
        raise RuntimeError("The SHA-256 of the registered ontology does not match the JSON-LD that was sent.")
    check("T8-4 The hash of the registered ontology matches", True, registered_ontology_hash)
    print("Failed item count of the latest indexer run:", indexer_client.get_indexer_status(resources["indexer"]).last_result.failed_item_count)
    display(pd.DataFrame([graph_stats]))

## 10. T9: Graph index

A graph index is an ordinary search index in which **one document = one edge (subject, predicate, object)**. The difference is `graph_profile`, which tells the service which field is the subject, the predicate, the object, the kind and the evidence chunk. GQL uses this mapping to traverse the index as a graph.

**In reuse mode, resume from this cell after the common setup and T1.** The notebook fetches the existing definition with `get_knowledge_source()` and restores the search target from `createdResources.graphIndex` and `graphExtraction.graphName`. The `resources` and `knowledge_source` objects from creation do not need to remain in the kernel. If the definition or the graph index cannot be retrieved, the notebook displays the error and stops.

`createdResources.graphIndex` may not be included in the response right after the knowledge source is created or updated. Even in new-build mode, the notebook fetches the latest definition again after the T8 sync completes. The `resources` variable that holds the update response is not updated automatically.

```mermaid
flowchart LR
    subgraph E1["Document 1 = edge 1"]
        S1["subject: Thornwick Castle"] -- "predicate: foundedBy" --> O1["object: Osric Thorne"]
    end
    subgraph EV["Evidence (provenance)"]
        CH["chunk_id: uid in the content index"]
    end
    E1 -. "chunk_id / parent_id" .-> EV
```

> Source: [Create and query an entity graph: re-fetching the generated resource names](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#create-the-graph-enabled-knowledge-source) / [GraphProfile / GraphFieldMappings(azure-search-documents 12.2.0b1)](https://pypi.org/project/azure-search-documents/12.2.0b1/) / [SearchIndexGraphDocuments.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchIndexGraphDocuments.json) / Hogan et al., [Knowledge Graphs](https://arxiv.org/abs/2003.02320), ACM Computing Surveys 54(4), 2021

In [ ]:
knowledge_source = index_client.get_knowledge_source(KS_NAME)
parameters = knowledge_source.azure_blob_parameters
if parameters is None or parameters.graph_extraction is None or not parameters.graph_extraction.graph_name:
    raise RuntimeError(f"{KS_NAME}: graphExtraction.graphName is missing. Specify an Entity Graph knowledge source.")
resources = parameters.created_resources or {}
graph_index_name = resources.get("graphIndex")
if not graph_index_name:
    raise RuntimeError(f"{KS_NAME}: createdResources.graphIndex is missing. Check the creation and sync state of the existing graph, and KS_NAME.")
GRAPH_NAME = parameters.graph_extraction.graph_name
graph_index = index_client.get_index(graph_index_name)
if graph_index.graph_profile is None:
    raise RuntimeError(f"{graph_index_name}: graph_profile is missing. Check the definition of the graph index.")
registered_ontology = json.loads(parameters.graph_extraction.ontology.value)
registered_terms = {item["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1] for item in registered_ontology["@graph"]}
sample_terms = set(classes["class"]) | set(relations["predicate"]) | set(datatype_properties["predicate"])
RUN_SAMPLE_TESTS = sample_terms <= registered_terms
if not RUN_SAMPLE_TESTS:
    print("T10 to T12 were not run: the types and relations of the built-in test are not all present, so check the registered contents in chapters 15 and 16.")
F = graph_index.graph_profile.fields
escaped_graph_name = GRAPH_NAME.replace("'", "''")
GRAPH_FILTER = f"{F.graph_name_field} eq '{escaped_graph_name}'"

graph_client = SearchClient(SEARCH_ENDPOINT, graph_index_name, credential, api_version=API_VERSION)
selectable = [f.name for f in graph_index.fields if f.retrievable is not False]
found = graph_client.search("*", filter=GRAPH_FILTER, select=selectable, include_total_count=True, top=1000)
edge_total = found.get_count()
edge_docs = pd.DataFrame(list(found))

check("T9-1 The graph profile is edgeList", getattr(graph_index.graph_profile.kind, "value", graph_index.graph_profile.kind) == "edgeList", graph_index.graph_profile.kind)
check("T9-2 The graph has edges", edge_total > 0, f"{edge_total} edges")
check("T9-3 Every edge has an evidence chunk attached", F.source_chunk_id_field in edge_docs and bool(edge_docs[F.source_chunk_id_field].astype(bool).all()), f"{int((~edge_docs[F.source_chunk_id_field].astype(bool)).sum())} edges without evidence")
display(pd.DataFrame([{"Role (graph_profile)": role, "Field name": field} for role, field in F.as_dict().items() if field]))
edge_docs.head(8)

## 11. T10: Expected facts (recall)

For each relation defined in the ontology (`domain`, `predicate`, `range`), the notebook runs a GQL `MATCH` and collects the edges. Labels (class names) and relation names are enclosed in backticks. GQL is the graph query language of ISO/IEC 39075: you write a pattern with `MATCH` and pick out values with `RETURN`. The notebook passes the query to `SearchClient.graph()` and specifies the graph-name filter in `filter`.

Next, it lists the **facts stated explicitly** in the articles as the "expected facts" and measures the share found among the edges (the **recall**). Names are compared after ignoring case, spacing and punctuation variants (such as hyphens and apostrophes), because variants like these can split one entity into separate nodes. LLM extraction varies, so the test passes when the recall is at least `MIN_RECALL`.

> Source: [GQL language guide for graph in Microsoft Fabric](https://learn.microsoft.com/fabric/graph/gql-language-guide) / [ISO/IEC 39075](https://www.iso.org/standard/76120.html) / [Graph traversal(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-consume/references/graph-traversal.md) / [sample_query_graph_preview.py](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/samples/sample_query_graph_preview.py)

In [ ]:
if RUN_SAMPLE_TESTS:
    frames = []
    for rel in relations.itertuples():
        result = graph_client.graph(gql=f"MATCH (a:`{rel.domain}`)-[r:`{rel.predicate}`]->(b:`{rel.range}`) RETURN a.name AS a, type(r) AS predicate, b.name AS b", filter=GRAPH_FILTER)
        frames.append(pd.DataFrame([row.as_dict() for row in result.value], columns=["a", "predicate", "b"]).assign(a_type=rel.domain, b_type=rel.range))
    edges = pd.concat(frames, ignore_index=True)

    EXPECTED = pd.DataFrame([
        ("Thornwick Castle", "foundedBy", ["Osric Thorne"]), ("Thornwick Castle", "locatedIn", ["Eastgate Ward", "Aldermere"]), ("Thornwick Castle", "hasFeature", ["Twelve Stars"]),
        ("Thornwick Castle", "survivedEvent", ["Great Fire"]), ("Thornwick Castle", "registeredAs", ["Historic Monuments"]),
        ("Elowen Park", "locatedIn", ["Eastgate Ward", "Aldermere"]), ("Elowen Park", "hasFeature", ["Elowen Cherry"]),
        ("Elowen Park", "survivedEvent", ["Great Fire"]), ("Elowen Park", "registeredAs", ["Historic Monuments"]),
        ("Wendell Hart", "visited", ["Moonwell", "Bindery"]),
    ], columns=["a", "predicate", "b_any"])
    EXPECTED["Found"] = [
        bool(((edges["predicate"] == e.predicate) & edges["a"].map(lambda x: norm(e.a) in norm(x)) & edges["b"].map(lambda x: any(norm(k) in norm(x) for k in e.b_any))).any())
        for e in EXPECTED.itertuples()
    ]
    recall = EXPECTED["Found"].mean()

    check("T10 The recall of the expected facts is at least MIN_RECALL", recall >= MIN_RECALL, f"{int(EXPECTED['Found'].sum())} / {len(EXPECTED)} = {recall:.0%} (minimum {MIN_RECALL:.0%})")
    print("Number of extracted edges:", len(edges))
    EXPECTED

### 11.1 Count per relation type, and a diagram of the whole graph

Node colors show the class (kind). You can see that the two articles share the nodes for the location, the World Heritage designation and the Great Fire of Aldermere, and are connected through them.

In [ ]:
if RUN_SAMPLE_TESTS:
    display(edges.groupby(["a_type", "predicate", "b_type"]).size().rename("Edge count").reset_index())

    instance_graph = nx.DiGraph()
    for e in edges.itertuples():
        instance_graph.add_node(e.a, type=e.a_type)
        instance_graph.add_node(e.b, type=e.b_type)
        instance_graph.add_edge(e.a, e.b, predicate=e.predicate)
    print("Number of nodes:", instance_graph.number_of_nodes(), "/ Number of edges:", instance_graph.number_of_edges())
    draw_graph(instance_graph, f"Knowledge graph '{GRAPH_NAME}'", figsize=(15, 11))

## 12. T11: Nodes shared across documents

The two articles state the same facts (the location, the event that both sites escaped, and the World Heritage designation). In the graph, **the same entity should become a single node, and both heritage sites should connect to it**. The notebook searches with a pattern that walks to a shared node from two directions. `a.name < b.name` is a condition that prevents the same pair from appearing twice in swapped order. A regular search has to pick the relevant sentences out of the two documents and compare them, but the graph expresses this in a single pattern.

In [ ]:
if RUN_SAMPLE_TESTS:
    shared = {}
    for predicate, cls in [("locatedIn", "Place"), ("survivedEvent", "Event"), ("registeredAs", "Designation")]:
        gql = (f"MATCH (a:HeritageSite)-[:{predicate}]->(n:{cls})<-[:{predicate}]-(b:HeritageSite) "
               "WHERE a.name < b.name RETURN a.name AS site_a, b.name AS site_b, n.name AS shared")
        shared[predicate] = rows(run_gql(gql))
        check(f"T11 {predicate}: the two heritage sites connect to the same node", len(shared[predicate]) >= 1, shared[predicate].to_dict("records")[:1])
    pd.concat({k: v for k, v in shared.items()})

## 13. T12: Multi-hop and provenance (tracing back to the evidence chunk)

The notebook solves "Who founded the castle that has a garden visited by Wendell Hart?" with a 3-hop pattern (`visited`, `hasFacility`, `foundedBy`). The query is written as a pattern without fixing the entity names. This makes an empty result less likely even if the spelling of the extracted entity names varies from run to run.

Each edge records the ID of the chunk it was extracted from (`chunk_id`). When you specify `provenance` in the query, the response returns, for each row, the evidence edges and the text of the chunks they came from.

- `kind="ids"`: returns only the IDs of the evidence edges and chunks.
- `kind="hydrate"`: also retrieves the chunk documents (specify the fields to return with `select`).

You use this to confirm the graph's answer against the original text.

> Source: [KnowledgeBaseRetrieveGraph.json / SearchIndexGraphDocuments.json](https://github.com/Azure/azure-rest-api-specs/tree/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples) / [sample_query_graph_preview.py](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/samples/sample_query_graph_preview.py)

In [ ]:
if RUN_SAMPLE_TESTS:
    MULTI_HOP = (
        "MATCH (p:Person)-[:visited]->(f:Facility)<-[:hasFacility]-(s:HeritageSite)-[:foundedBy]->(g:Person) "
        "RETURN p.name AS visitor, f.name AS facility, s.name AS site, g.name AS founder"
    )
    result = run_gql(MULTI_HOP, provenance=GraphQueryProvenanceParameters(
        kind="hydrate", max_evidence_per_row=3, hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(select="uid,snippet")))
    hop = rows(result)

    evidence = []
    for row in result.value:
        d = row.as_dict()
        for ev in d["@search.graphProvenance"]["evidence"]:
            doc = result.provenance_documents.get(ev["chunkId"], {})
            evidence.append({"facility": d["facility"], "site": d["site"], "founder": d["founder"], "chunk_id": ev["chunkId"], "Evidence chunk": (doc.get("snippet") or "")[:140]})
    evidence_df = pd.DataFrame(evidence, columns=["facility", "site", "founder", "chunk_id", "Evidence chunk"])

    check("T12-1 The 3-hop result includes the founder (Osric Thorne)", len(hop) >= 1 and hop["founder"].map(lambda x: norm("Osric Thorne") in norm(x)).any(), hop.to_dict("records")[:1])
    check("T12-2 The text of the evidence chunks can be retrieved", len(evidence_df) >= 1 and bool(evidence_df["Evidence chunk"].str.len().gt(0).all()), f"{len(evidence_df)} evidence items")

    path_graph = nx.DiGraph()
    for r in hop.itertuples():
        path_graph.add_node(r.visitor, type="Person")
        path_graph.add_node(r.founder, type="Person")
        path_graph.add_node(r.facility, type="Facility")
        path_graph.add_node(r.site, type="HeritageSite")
        path_graph.add_edge(r.visitor, r.facility, predicate="visited")
        path_graph.add_edge(r.site, r.facility, predicate="hasFacility")
        path_graph.add_edge(r.site, r.founder, predicate="foundedBy")
    if len(hop):
        draw_graph(path_graph, "Multi-hop search result", figsize=(10, 6))
    evidence_df

## 14. T13: Knowledge base (optional)

A knowledge base bundles knowledge sources. For a question, the LLM builds a retrieval plan, uses graph search (generating and running GQL) and text search as appropriate, and synthesizes an answer with citations. **Because it calls an LLM, this test runs only when `RUN_KB_TEST = True`.** It checks the following.

- The answer text and the references (the targets that `[ref_id:N]` points to) are returned.
- The execution log (`activity`) includes the graph search (`graphSearch`).

If the graph search returns nothing, the knowledge base tries to answer from the text search results. When the spelling of an entity name in the question differs from the node name in the graph (for example in case, spacing or punctuation), the graph search may return `EmptyResults`. The notebook therefore uses a question that is less affected by spelling variants.

> Source: [Query a knowledge base using the retrieve action](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve) / [KnowledgeBaseRetrieveGraph.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
from azure.search.documents.indexes.models import AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel, KnowledgeSourceReference
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    AgentRuntimeGraphHydrateDocumentsParams,
    AgentRuntimeGraphParams,
    AgentRuntimeGraphProvenanceParams,
    AzureBlobKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

if RUN_KB_TEST:
    chat_params = AzureOpenAIVectorizerParameters(resource_url=AOAI_ENDPOINT, deployment_name=CHAT_DEPLOYMENT, model_name=CHAT_MODEL)
    index_client.create_or_update_knowledge_base(KnowledgeBase(
        name=KB_NAME,
        description="A knowledge graph and text chunks built from Hanapedia articles about heritage sites (Thornwick Castle and Elowen Park).",
        knowledge_sources=[KnowledgeSourceReference(name=KS_NAME)],
        models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=chat_params)],
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        retrieval_instructions=("Use *_graph_search tools for structured questions about entities, their properties, and relationships, including finding entities that meet "
                                "specific criteria or tracing connections. Use relevance-based search tools to find information by topic, meaning, or wording. "
                                "Express graph requests as natural-language questions describing the entities, relationships, and criteria sought."),
        answer_instructions=("Answer only from the retrieved graph rows and passages. Cite every claim with its [ref_id:N]. Do not add facts that the retrieved results do not state."),
    ))
    kb_client = KnowledgeBaseRetrievalClient(SEARCH_ENDPOINT, credential, knowledge_base_name=KB_NAME)
    answer = kb_client.retrieve(KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text="Which event did both Thornwick Castle and Elowen Park escape without being destroyed by fire?")])],
        include_activity=True, output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS, max_runtime_in_seconds=120,
        knowledge_source_params=[AzureBlobKnowledgeSourceParams(
            knowledge_source_name=KS_NAME, include_references=True, include_reference_source_data=True,
            graph_parameters=AgentRuntimeGraphParams(include_query=True, provenance=AgentRuntimeGraphProvenanceParams(
                kind="hydrate", hydrate_documents=AgentRuntimeGraphHydrateDocumentsParams(top=10))))],
    ))
    activity = pd.DataFrame([{"Type": a.as_dict()["type"], "Result": (a.as_dict().get("graphSearchArguments") or {}).get("outcome"),
                              "GQL": (a.as_dict().get("graphSearchArguments") or {}).get("gql")} for a in answer.activity])
    check("T13-1 An answer and references are returned", bool(answer.response[0].content[0].text) and len(answer.references) >= 1, f"{len(answer.references)} references")
    check("T13-2 The execution log includes a graph search", (activity["Type"] == "graphSearch").any(), activity["Type"].tolist())
    print("Answer:", answer.response[0].content[0].text)
    display(activity)

## 15. Read back the registered contents of Foundry IQ and visualize them

The notebook reads the knowledge source created or updated in T5 and T8, or the existing source named by `KS_NAME`, as it is. This chapter does not create a new source, re-ingest data or call a model. Even in a new kernel, you can resume with **the settings in 1.2 (`RUN_BUILD = False`) → the drawing helpers in 1.5 → T1 → this chapter**.

The diagrams are based on the registered contents re-fetched from Foundry IQ, not on design data in the notebook. The notebook displays a subgraph of at most `MAX_DISPLAY_EDGES` documents.

> Source: [Creating an Entity Graph and querying it directly](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph)

### 15.1 Registered ontology

The notebook fetches the JSON-LD, the graph name and the generated index names from the latest definition of the knowledge source. It is a blueprint in which classes are nodes and object properties are arrows. The displayed hash identifies the current registered contents. T8 verifies that it matches what was sent.

> Source: [Knowledge Sources - Get](https://learn.microsoft.com/rest/api/searchservice/knowledge-sources/get?view=rest-searchservice-2026-11-01-preview) / [OWL 2 Primer](https://www.w3.org/TR/owl2-primer/)

In [ ]:
ENTITY_KS_NAME = KS_NAME
registered_source = index_client.get_knowledge_source(ENTITY_KS_NAME)
parameters = registered_source.azure_blob_parameters
if parameters is None or parameters.graph_extraction is None or parameters.graph_extraction.ontology is None:
    raise ValueError("A registered graphExtraction and ontology are required.")
ENTITY_GRAPH_NAME = parameters.graph_extraction.graph_name
resources = parameters.created_resources or {}
registered_ontology_text = parameters.graph_extraction.ontology.value
registered_ontology_hash = hashlib.sha256(registered_ontology_text.encode("utf-8")).hexdigest()
registered_ontology = json.loads(registered_ontology_text)
print("SHA-256 of the registered ontology:", registered_ontology_hash)
display(pd.DataFrame([{"Knowledge source": ENTITY_KS_NAME, "Graph name": ENTITY_GRAPH_NAME, "Graph index": resources.get("graphIndex"), "Content index": resources.get("index")}]))

In [ ]:
ontology_graph = nx.MultiDiGraph()
for declaration in registered_ontology["@graph"]:
    if declaration["@type"] == "owl:Class":
        ontology_graph.add_node(declaration["@id"], label=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]), kind="Class")
for declaration in registered_ontology["@graph"]:
    if declaration["@type"] == "owl:ObjectProperty":
        ontology_graph.add_edge(declaration["rdfs:domain"]["@id"], declaration["rdfs:range"]["@id"],
                                predicate=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]))
draw_graph(ontology_graph, "Ontology registered in Foundry IQ: classes and relations")

## 15.2 Visualize the extracted entities and relationships

The notebook uses the `createdResources.graphIndex` obtained in 15.1 and does not run the wait cell. If the graph index does not exist yet, the notebook stops explicitly. After the extraction has progressed, re-run from the definition fetch in 15.1.

The field names come from `graph_profile.fields` of the registered index. The diagram is limited to relationships between entities that have an `object_id`. You can check attributes such as strings and numbers in the table of retrieved documents. The diagram is a subgraph within the retrieval limit and does not always show the whole graph. If you run it during extraction, it shows the data registered up to that point.

> Source: [Entity Graph index and provenance](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
entity_index_name = resources.get("graphIndex")
if not entity_index_name:
    raise RuntimeError("The knowledge source's createdResources has no graphIndex. Check the extraction state and re-run from the definition fetch in 15.1.")
entity_index = index_client.get_index(entity_index_name)
F = entity_index.graph_profile.fields
entity_client = SearchClient(SEARCH_ENDPOINT, entity_index_name, credential, api_version=API_VERSION)
entity_select = [field.name for field in entity_index.fields if field.retrievable is not False]
escaped_graph_name = ENTITY_GRAPH_NAME.replace("'", "''")


entity_found = entity_client.search("*", filter=f"{F.graph_name_field} eq '{escaped_graph_name}'", select=entity_select,
                                    include_total_count=True, top=MAX_DISPLAY_EDGES)
entity_total = entity_found.get_count()
entity_frame = pd.DataFrame(list(entity_found))
if entity_frame.empty:
    raise RuntimeError("The specified graph has no edges. Check the extraction state, the ontology and the indexer results.")
entity_relations = entity_frame[entity_frame[F.object_id_field].fillna("").ne("")]
if entity_relations.empty:
    raise RuntimeError("There are no relationships between entities in the retrieved range. Check the limit and the ontology.")
print(f"{entity_total} documents in the graph / {len(entity_frame)} retrieved / {len(entity_relations)} relationships in the diagram")
display(entity_frame[[F.subject_id_display_name_field, F.predicate_field, F.object_id_display_name_field, F.object_string_field, F.source_chunk_id_field]].head(12))

In [ ]:
entity_graph = nx.MultiDiGraph()
for row in entity_relations.to_dict("records"):
    entity_graph.add_node(row[F.subject_id_field], label=row[F.subject_id_display_name_field], kind=row[F.subject_type_field])
    entity_graph.add_node(row[F.object_id_field], label=row[F.object_id_display_name_field], kind=row[F.object_type_field])
    entity_graph.add_edge(row[F.subject_id_field], row[F.object_id_field], predicate=row[F.predicate_field])
draw_graph(entity_graph, f"Real data in Foundry IQ: {entity_index_name}")

### Check the evidence for one relationship

The notebook links the chunk ID of an extraction result to the content index of the same knowledge source. It checks not only that a triple exists but also which text it is based on. It does not fill in facts that are absent from the articles based only on the layout of the visualization or the model's output.

> Source: [Graph provenance and source chunks](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
edge = entity_relations.iloc[0]
chunk_id = edge[F.source_chunk_id_field]
if not isinstance(chunk_id, str) or not chunk_id:
    raise RuntimeError("The selected relationship has no evidence chunk ID.")
content_client = SearchClient(SEARCH_ENDPOINT, resources["index"], credential, api_version=API_VERSION)
evidence = content_client.get_document(chunk_id, selected_fields=["uid", "blob_url", "snippet"])
display(pd.DataFrame([{"Subject": edge[F.subject_id_display_name_field], "Relation": edge[F.predicate_field], "Object": edge[F.object_id_display_name_field]}]))
display(pd.DataFrame([evidence]))

In [ ]:
from azure.search.documents.models import (
    GraphQueryResult,
    GraphQueryProvenanceParameters,
    GraphQueryProvenanceHydrateDocumentsParameters,
)


def display_gql(result: GraphQueryResult) -> None:
    print("Returned rows:", len(result.value))
    display(pd.json_normalize(result.as_dict()["value"]))
    if result.diagnostics:
        print("Diagnostics (warnings, errors and execution info):")
        display(pd.DataFrame([item.as_dict() for item in result.diagnostics]))
    if result.statistics:
        print("Execution statistics:")
        display(pd.DataFrame([result.statistics.as_dict()]))

## 16 Exploration, aggregation and evidence checks with GQL

Run the GQL display function at the end of chapter 15 and the cell in 15.1 that fetches the registered definition before you continue. You do not need the drawing cell in 15.1, the document search, drawing and evidence retrieval in 15.2, or other notebooks. The checks go in this order: types and relations → names → 2-hop paths → aggregation by location → evidence text.

> Source: [Querying an Entity Graph directly with GQL](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

### 16.0 Prepare the graph client

Run this after you fetch the latest knowledge source definition in 15.1. You do not need the drawing cells or T10 to T12. The queries below use `HeritageSite`, `Feature`, `Facility` and `Place` and the relation names of the built-in ontology. For an external JSON-LD, change the queries to match the definition in 15.1. GQL does not call a model.

> Source: [Running GQL directly](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
gql_entity_index_name = resources.get("graphIndex")
if not gql_entity_index_name:
    raise RuntimeError("createdResources has no graphIndex. Check the extraction state and re-run from the definition fetch in 15.1.")
gql_entity_index = index_client.get_index(gql_entity_index_name)
if gql_entity_index.graph_profile is None or gql_entity_index.graph_profile.kind != "edgeList":
    raise ValueError("GQL requires an edgeList graphProfile.")
gql_graph_field = gql_entity_index.graph_profile.fields.graph_name_field
gql_graph_name = ENTITY_GRAPH_NAME.replace("'", "''")
entity_gql_filter = f"{gql_graph_field} eq '{gql_graph_name}'"
entity_gql_client = SearchClient(SEARCH_ENDPOINT, gql_entity_index_name, credential, api_version=API_VERSION)

### 16.1 Filter by type and relation

Retrieve the `hasFeature` relations from `HeritageSite` to `Feature`. The type label and the predicate are separate conditions.

> Source: [GQL that specifies types and relations](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
entity_relations_gql = """
MATCH (site:HeritageSite)-[edge]->(feature:Feature)
WHERE edge.predicate = 'hasFeature'
RETURN site.id AS site_id, site.name AS site, feature.name AS feature
ORDER BY site, feature
LIMIT 10
"""
entity_relations_result = entity_gql_client.graph(
    gql=entity_relations_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_relations_result)

### 16.2. Explore starting from a name

Retrieve the outgoing relations from the node whose display name is "Thornwick Castle". The result also includes the IDs, so you can tell apart different nodes that have the same name. To narrow the query to a specific ID, replace `subject.name = 'Thornwick Castle'` with `subject.id = '<the ID you retrieved>'`. If multiple values come back, check them against the evidence in 16.5.

> Source: [Filtering by name](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Example of filtering by ID](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
entity_name_gql = """
MATCH (subject)-[edge]->(object)
WHERE subject.name = 'Thornwick Castle'
RETURN subject.id AS subject_id, subject.name AS subject_name,
       edge.predicate AS relation, object.id AS object_id, object.name AS object_name
ORDER BY relation, object_name
LIMIT 10
"""
entity_name_result = entity_gql_client.graph(
    gql=entity_name_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_name_result)

### 16.3 Walk from facility to visitor in 2 hops

Retrieve the path `heritage site → hasFacility → facility in the grounds ← visited ← person`. It is a single query that shares the intermediate facility.

> Source: [Multi-hop GQL](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
entity_path_gql = """
MATCH (site)-[first]->(facility)<-[second]-(person)
WHERE first.predicate = 'hasFacility' AND second.predicate = 'visited'
RETURN site.name AS site, facility.name AS facility, person.name AS visitor
ORDER BY site, facility, visitor
LIMIT 10
"""
entity_path_result = entity_gql_client.graph(
    gql=entity_path_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_path_result)

### 16.4 Aggregate the number of relations per location

Aggregate the `locatedIn` paths by place ID and display name. `relation_count` is the number of matched path rows. It is not necessarily the number of evidence documents or the number of deduplicated facilities.

> Source: [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
entity_count_gql = """
MATCH (site)-[edge]->(place)
WHERE edge.predicate = 'locatedIn'
RETURN place.id AS place_id, place.name AS place, count(*) AS relation_count
GROUP BY place.id, place.name
ORDER BY relation_count DESC, place
LIMIT 10
"""
entity_count_result = entity_gql_client.graph(
    gql=entity_count_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_count_result)

### 16.5. Retrieve GQL results together with the evidence text

When you set `provenance.kind = hydrate`, the `@search.graphProvenance` of each result row carries the IDs of the edge, chunk and parent document, and the text comes back together in `@search.graphProvenanceDocuments` for the whole response. The content index it reads from is the one specified by `graphProfile.provenanceIndexName`. The Python side does not reuse the search results of 15.2.

`max_evidence_per_row = 3` is the upper limit on the number of evidence items retrieved per row. If `evidenceCount.relation` is `exact`, the count is complete, and if it is `lowerBound`, more evidence may exist. GQL also works with a Topic Graph, but the Topic Graph index in this release has no `provenanceIndexName`, so the automatic retrieval of the text shown here runs against a managed Entity Graph.

> Source: [GQL and automatic retrieval of evidence documents](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Provenance in Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if not gql_entity_index.graph_profile.provenance_index_name:
    raise ValueError("graphProfile.provenanceIndexName is required to retrieve the text automatically.")
entity_evidence_gql = """
MATCH (site)-[edge]->(feature)
WHERE site.name = 'Thornwick Castle' AND edge.predicate = 'hasFeature'
RETURN site.name AS site, feature.name AS feature
ORDER BY feature
LIMIT 10
"""
entity_evidence_result = entity_gql_client.graph(
    gql=entity_evidence_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
    provenance=GraphQueryProvenanceParameters(
        kind="hydrate", max_evidence_per_row=3,
        hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(select="uid,blob_url,snippet"),
    ),
)
display_gql(entity_evidence_result)
display(pd.DataFrame.from_dict(
    entity_evidence_result.as_dict()["@search.graphProvenanceDocuments"], orient="index",
).rename_axis("Evidence chunk ID"))

In [ ]:
check("T14-1 The registered ontology can be retrieved", bool(registered_ontology["@graph"]), registered_ontology_hash)
check("T14-2 The registered relationships between entities can be retrieved", not entity_relations.empty, f"{len(entity_relations)} relationships drawn")
check("T14-3 The evidence text can be retrieved", bool(evidence.get("snippet")), edge[F.source_chunk_id_field])
display(pd.DataFrame([{"Knowledge source": KS_NAME, "Graph index": entity_index_name,
                       "Documents in the graph": entity_total, "Retrieved count": len(entity_frame)}]))

## 17. Results list

The notebook aggregates only the tests that ran and displays the number of PASS and FAIL results and the details of the FAILs. In reuse mode, it does not treat T2 to T8 as passed and does not include them in the list. LLM extraction varies, so when T10 or T12 is FAIL, re-run the test and compare the recall and the extracted edges. Spelling variants of names (differences in case, spacing or punctuation) can also split nodes.

In [ ]:
summary = pd.DataFrame(results)
print(summary["Result"].value_counts().to_dict())
summary.style.apply(lambda row: ["background-color: #ffd6d6" if row["Result"] == "FAIL" else "" for _ in row], axis=1)

## 18. Cleanup (optional)

**Only when you set `RUN_CLEANUP = True` in new-build mode**, the notebook deletes the knowledge base and the knowledge source. If you request deletion in reuse mode, the notebook stops with an error to protect the existing resources. Deleting the knowledge source also deletes the automatically generated data source, indexer, skillset, content index and graph index.

The following are not deleted, so delete them separately on the storage side if you no longer need them.

- The containers and tables that start with `ms-az-search-indexercache-` in the cache storage
- The Blob container (`ft-entity`) in the source storage, and the uploaded files

In [ ]:
if RUN_CLEANUP and not RUN_BUILD:
    raise ValueError("Reuse mode does not delete existing resources. Set RUN_CLEANUP = False.")

if RUN_CLEANUP:
    for delete, name in ((index_client.delete_knowledge_base, KB_NAME), (index_client.delete_knowledge_source, KS_NAME)):
        try:
            delete(name)
            print("Deleted:", name)
        except ResourceNotFoundError:
            pass